In [20]:
import sys
sys.path.append("../")
from src.embeddings import TransformerEmbeddings
from src.model import MultiHeadAttention
import torch 
import torch.nn as nn


In [25]:
# create a dummy input for tensor shape pass check 
def get_random_embedding(vocab_size=100, d_model=64):

    x = torch.randint(0, vocab_size, (8, 10)) # (batch_size, seq_len)
    embeddings = TransformerEmbeddings(vocab_size, d_model)
    x = embeddings(x) # (batch_size, seq_len, d_model)
    return x

In [ ]:
Q = get_random_embedding()
K = get_random_embedding()
V = get_random_embedding()

print(f"shape of Q: {Q.shape}\n") # (8, 10, 64)

mha = MultiHeadAttention(d_model=64, n_heads=4)
output = mha(Q, K, V)

print(f"shape of output after mha: {output.shape}") # (8, 10, 64)

shape of Q: torch.Size([8, 10, 64])

shape of output after mha: torch.Size([8, 10, 64])


In [27]:
# test for masked attention

batch_size, seq_len, d_model, n_heads = 8, 10, 64, 4

Q = get_random_embedding(vocab_size=100, d_model=d_model)
K = get_random_embedding(vocab_size=100, d_model=d_model)
V = get_random_embedding(vocab_size=100, d_model=d_model)

# Causal mask: lower-triangular matrix of 1s
# Shape: (1, 1, seq_len, seq_len) — broadcasts over batch & heads
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
print(f"Causal mask shape: {causal_mask.shape}")
print(f"Causal mask:\n{causal_mask[0, 0]}\n")

mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)

# Without mask — all positions can attend to each other
output_no_mask = mha(Q, K, V)
print(f"Output (no mask) shape: {output_no_mask.shape}")

# With causal mask — position i can only attend to positions 0..i
output_masked = mha(Q, K, V, mask=causal_mask)
print(f"Output (causal mask) shape: {output_masked.shape}")

# They should differ since masking changes the attention pattern
print(f"\nOutputs are different: {not torch.allclose(output_no_mask, output_masked)}")


Causal mask shape: torch.Size([1, 1, 10, 10])
Causal mask:
tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])

Output (no mask) shape: torch.Size([8, 10, 64])
Output (causal mask) shape: torch.Size([8, 10, 64])

Outputs are different: True


In [2]:
# full pipeline dimension check for Transformer class.. import torch
import sys
import os
import torch 
# Add the src directory to the path so we can import our modules
sys.path.append(os.path.abspath('../'))

from src.architecture import Transformer
from src.utils import make_src_mask, make_tgt_mask

def run_smoke_test():
    # 1. Hyperparameters
    vocab_size = 20      # 10 digits + special tokens
    d_model = 128
    n_layers = 2
    n_heads = 8
    d_ff = 512
    dropout = 0.1
    max_seq_len = 50
    pad_idx = 0
    
    batch_size = 8
    src_len = 10
    tgt_len = 10

    print(f"--- Initializing Transformer Model ---")
    model = Transformer(
        src_vocab_size=vocab_size,
        tgt_vocab_size=vocab_size,
        d_model=d_model,
        n_layers=n_layers,
        n_heads=n_heads,
        d_ff=d_ff,
        dropout=dropout,
        max_seq_len=max_seq_len
    )

    # 2. Mock Data (Random integers representing token IDs)
    src = torch.randint(1, vocab_size, (batch_size, src_len))
    tgt = torch.randint(1, vocab_size, (batch_size, tgt_len))
    
    # 3. Generate Masks using your utils
    src_mask = make_src_mask(src, pad_idx)
    tgt_mask = make_tgt_mask(tgt, pad_idx)

    print(f"Input Shapes: src={src.shape}, tgt={tgt.shape}")
    print(f"Mask Shapes:  src_mask={src_mask.shape}, tgt_mask={tgt_mask.shape}")

    # 4. Forward Pass
    try:
        model.eval() # Set to evaluation mode
        with torch.no_grad():
            output = model(src, tgt, src_mask, tgt_mask)
        
        print(f"--- Success! ---")
        print(f"Output Shape: {output.shape}")
        
        # Expected shape check
        expected_shape = (batch_size, tgt_len, vocab_size)
        if output.shape == expected_shape:
            print(f"✅ Dimensions match: {output.shape}")
        else:
            print(f"❌ Dimension mismatch! Expected {expected_shape}, got {output.shape}")

    except Exception as e:
        print(f"--- Failure! ---")
        print(f"An error occurred during the forward pass:")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    run_smoke_test()


--- Initializing Transformer Model ---
Input Shapes: src=torch.Size([8, 10]), tgt=torch.Size([8, 10])
Mask Shapes:  src_mask=torch.Size([8, 1, 1, 10]), tgt_mask=torch.Size([8, 1, 10, 10])
--- Success! ---
Output Shape: torch.Size([8, 10, 20])
✅ Dimensions match: torch.Size([8, 10, 20])
